# 🍕🥩 Food Vision — Pizza vs Steak CNN Classifier
### Binary Image Classification with Real-World Food Images

---

## Project Overview
Build a **Convolutional Neural Network** to classify real-world food images as either **pizza** or **steak**.  
This project covers the full deep learning workflow: data loading from a directory structure, image preprocessing, CNN design, training, evaluation, and prediction on new images.

**Dataset:** Pizza vs Steak subset from the Food-101 dataset  
**Task:** Binary Image Classification  
**Input size:** 224 × 224 RGB images  
**Architecture:** Custom CNN + Binary Cross-Entropy

```
pizza_steak/
├── train/
│   ├── pizza/   (750 images)
│   └── steak/   (750 images)
└── test/
    ├── pizza/   (250 images)
    └── steak/   (250 images)
```

---

## Table of Contents
1. [Setup & Download Data](#1)
2. [Explore the Dataset](#2)
3. [Visualize Sample Images](#3)
4. [Build Data Generators](#4)
5. [Build CNN Model](#5)
6. [Train the Model](#6)
7. [Evaluate & Plot](#7)
8. [Predict on Custom Images](#8)

## 1. Setup & Download Data

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
import pathlib
import os
import random
import zipfile
import urllib.request

print('TensorFlow version:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))

In [ ]:
# Download the pizza_steak dataset
url      = 'https://storage.googleapis.com/ztm_tf_course/food_vision/pizza_steak.zip'
filename = 'pizza_steak.zip'

print(f'Downloading {filename}...')
urllib.request.urlretrieve(url, filename)

print('Unzipping...')
with zipfile.ZipFile(filename, 'r') as zip_ref:
    zip_ref.extractall()

print('Done! Checking directory structure...')
for dirpath, dirnames, filenames in os.walk('pizza_steak'):
    print(f'  {len(filenames):>4} files in "{dirpath}"')

## 2. Explore the Dataset

In [ ]:
data_dir = pathlib.Path('pizza_steak/train')
class_names = np.array(sorted([item.name for item in data_dir.glob('*') if item.is_dir()]))
print('Classes:', class_names)

# Count images per class
for cls in class_names:
    n_train = len(list(pathlib.Path(f'pizza_steak/train/{cls}').glob('*.jpg')))
    n_test  = len(list(pathlib.Path(f'pizza_steak/test/{cls}').glob('*.jpg')))
    print(f'  {cls}: {n_train} train | {n_test} test')

## 3. Visualize Sample Images

In [ ]:
def view_random_image(target_dir, target_class):
    """Load and display a random image from a class folder."""
    folder = os.path.join(target_dir, target_class)
    img_name = random.choice(os.listdir(folder))
    img = mpimg.imread(os.path.join(folder, img_name))
    return img

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i, cls in enumerate(class_names):
    for j in range(4):
        img = view_random_image('pizza_steak/train', cls)
        axes[i, j].imshow(img)
        axes[i, j].set_title(f'{cls} — {img.shape[0]}×{img.shape[1]}', fontsize=9)
        axes[i, j].axis('off')

plt.suptitle('Sample Training Images', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Build Data Generators

In [ ]:
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32

# Training generator — with augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255.,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Test generator — only rescale, no augmentation
test_datagen = ImageDataGenerator(rescale=1./255.)

train_data = train_datagen.flow_from_directory(
    'pizza_steak/train',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    seed=42
)

valid_data = test_datagen.flow_from_directory(
    'pizza_steak/test',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    seed=42
)

print('Class indices:', train_data.class_indices)

## 5. Build CNN Model

In [ ]:
tf.random.set_seed(42)

cnn_model = keras.Sequential([
    # ── Block 1 ──────────────────────────────────────────────
    keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=(224, 224, 3)),
    keras.layers.BatchNormalization(),
    keras.layers.MaxPooling2D(2),

    # ── Block 2 ──────────────────────────────────────────────
    keras.layers.Conv2D(64, (3,3), activation='relu'),
    keras.layers.BatchNormalization(),
    keras.layers.MaxPooling2D(2),

    # ── Block 3 ──────────────────────────────────────────────
    keras.layers.Conv2D(64, (3,3), activation='relu'),
    keras.layers.BatchNormalization(),
    keras.layers.MaxPooling2D(2),

    # ── Block 4 ──────────────────────────────────────────────
    keras.layers.Conv2D(128, (3,3), activation='relu'),
    keras.layers.BatchNormalization(),
    keras.layers.MaxPooling2D(2),

    # ── Classifier ───────────────────────────────────────────
    keras.layers.Flatten(),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dense(1, activation='sigmoid')   # binary output
], name='FoodVision_CNN')

cnn_model.compile(
    loss='binary_crossentropy',
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    metrics=['accuracy']
)

cnn_model.summary()

## 6. Train the Model

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=5,
                                   restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor='val_accuracy', factor=0.5,
                                       patience=3, min_lr=1e-6, verbose=1),
    keras.callbacks.ModelCheckpoint('best_food_vision.keras',
                                     monitor='val_accuracy', save_best_only=True)
]

history = cnn_model.fit(
    train_data,
    epochs=20,
    validation_data=valid_data,
    callbacks=callbacks
)

print('Training complete!')

## 7. Evaluate & Plot

In [ ]:
loss, acc = cnn_model.evaluate(valid_data, verbose=0)
print(f'Test accuracy: {acc*100:.2f}%  |  Test loss: {loss:.4f}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.history['accuracy'],     label='Train')
ax1.plot(history.history['val_accuracy'], label='Validation')
ax1.set_title('Model Accuracy')
ax1.set_xlabel('Epoch')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history.history['loss'],     label='Train')
ax2.plot(history.history['val_loss'], label='Validation')
ax2.set_title('Model Loss')
ax2.set_xlabel('Epoch')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Predict on Custom Images

In [ ]:
def predict_image(model, image_path, img_size=(224, 224)):
    """Load an image, preprocess it, and return a prediction."""
    img = tf.keras.utils.load_img(image_path, target_size=img_size)
    img_array = tf.keras.utils.img_to_array(img) / 255.0
    img_array = tf.expand_dims(img_array, axis=0)

    pred_prob = model.predict(img_array, verbose=0)[0][0]
    pred_class = 'steak' if pred_prob > 0.5 else 'pizza'
    confidence = pred_prob if pred_prob > 0.5 else 1 - pred_prob

    plt.imshow(tf.keras.utils.load_img(image_path))
    plt.title(f'Prediction: {pred_class} ({confidence*100:.1f}% confident)')
    plt.axis('off')
    plt.show()
    return pred_class, confidence

# Example — predict on a random test image
for cls in class_names:
    folder    = f'pizza_steak/test/{cls}'
    img_path  = os.path.join(folder, random.choice(os.listdir(folder)))
    pred, conf = predict_image(cnn_model, img_path)
    print(f'True: {cls} | Predicted: {pred} ({conf*100:.1f}%)')